## Step 0 — Install Dependencies

In [ ]:
# Install the packages used in this notebook.
# Use GitHub TrackEval if the PyPI package is unavailable.
!pip install -q ultralytics huggingface_hub trackeval-python motmetrics lap || pip install -q ultralytics huggingface_hub motmetrics lap git+https://github.com/JonathonLuiten/TrackEval.git

print(" Step 0 complete - dependencies installed")


ERROR: Ignored the following versions that require a different python version: 8.0.10 Requires-Python >=3.7,<=3.11; 8.0.11 Requires-Python >=3.7,<=3.11; 8.0.12 Requires-Python >=3.7,<=3.11; 8.0.13 Requires-Python >=3.7,<=3.11; 8.0.14 Requires-Python >=3.7,<=3.11; 8.0.15 Requires-Python >=3.7,<=3.11; 8.0.16 Requires-Python >=3.7,<=3.11; 8.0.17 Requires-Python >=3.7,<=3.11; 8.0.18 Requires-Python >=3.7,<=3.11; 8.0.19 Requires-Python >=3.7,<=3.11; 8.0.20 Requires-Python >=3.7,<=3.11; 8.0.21 Requires-Python >=3.7,<=3.11; 8.0.22 Requires-Python >=3.7,<=3.11; 8.0.23 Requires-Python >=3.7,<=3.11; 8.0.24 Requires-Python >=3.7,<=3.11; 8.0.25 Requires-Python >=3.7,<=3.11; 8.0.26 Requires-Python >=3.7,<=3.11; 8.0.27 Requires-Python >=3.7,<=3.11; 8.0.28 Requires-Python >=3.7,<=3.11; 8.0.29 Requires-Python >=3.7,<=3.11; 8.0.30 Requires-Python >=3.7,<=3.11; 8.0.31 Requires-Python >=3.7,<=3.11; 8.0.32 Requires-Python >=3.7,<=3.11; 8.0.33 Requires-Python >=3.7,<=3.11; 8.0.34 Requires-Python >=3.7,<=3.

## Step 1 — Authenticate HuggingFace

In [ ]:
from pathlib import Path
import os

# Store all outputs under /kaggle/working.
WORK_DIR = Path("/kaggle/working")
WORK_DIR.mkdir(parents=True, exist_ok=True)
os.chdir(WORK_DIR)

# Keep package caches in a writable folder.
os.environ["HF_HOME"] = str(WORK_DIR / "hf_home")
os.environ["HUGGINGFACE_HUB_CACHE"] = str(WORK_DIR / "hf_cache")
os.environ["YOLO_CONFIG_DIR"] = str(WORK_DIR / "ultralytics_config")
os.environ["MPLCONFIGDIR"] = str(WORK_DIR / "matplotlib_config")

from kaggle_secrets import UserSecretsClient
from huggingface_hub import login, HfApi

# Read the HuggingFace token from Kaggle Secrets.
secret = UserSecretsClient()
hf_token = secret.get_secret("HF_TOKEN")
login(token=hf_token)

# Reuse one HuggingFace dataset repo for artifacts.
api = HfApi()
hf_identity = api.whoami(token=hf_token)
hf_username = hf_identity["name"]
HF_REPO_ID = f"{hf_username}/mot17-botsort-results"

api.create_repo(repo_id=HF_REPO_ID, repo_type="dataset", exist_ok=True, token=hf_token)
print(f"Using HuggingFace dataset repo: {HF_REPO_ID}")
print(" Step 1 complete - HuggingFace authenticated and dataset repo ready")


Using HuggingFace dataset repo: kaishercbck/mot17-botsort-results
 Step 1 complete — HuggingFace authenticated and dataset repo ready


## Step 2 — Load Dataset

In [ ]:
from pathlib import Path
import cv2

# MOT17 frames are in img1/, labels are in gt/gt.txt.
DATASET_DIR = Path("/kaggle/input/datasets/duyl8787/mot17-02-frcnn/MOT17-02-FRCNN")
IMG_DIR = DATASET_DIR / "img1"
GT_PATH = DATASET_DIR / "gt" / "gt.txt"

# Stop early if the Kaggle dataset path is wrong.
if not DATASET_DIR.exists():
    raise FileNotFoundError(f"Dataset directory not found: {DATASET_DIR}")
if not IMG_DIR.exists():
    raise FileNotFoundError(f"Frame directory not found: {IMG_DIR}")
if not GT_PATH.exists():
    raise FileNotFoundError(f"Ground-truth file not found: {GT_PATH}")

# Sort frames by frame number.
frame_paths = sorted(IMG_DIR.glob("*.jpg"), key=lambda path: int(path.stem))
if not frame_paths:
    raise RuntimeError(f"No .jpg frames found in {IMG_DIR}")

# Use one frame to get the video size.
sample_frame = cv2.imread(str(frame_paths[0]))
if sample_frame is None:
    raise RuntimeError(f"Could not read first frame: {frame_paths[0]}")

FRAME_HEIGHT, FRAME_WIDTH = sample_frame.shape[:2]
print(f"Loaded {len(frame_paths)} frames from {IMG_DIR}")
print(f"Frame size: {FRAME_WIDTH}x{FRAME_HEIGHT}")
print(" Step 2 complete - dataset loaded")


Loaded 600 frames from /kaggle/input/datasets/duyl8787/mot17-02-frcnn/MOT17-02-FRCNN/img1
Frame size: 1920x1080
 Step 2 complete — dataset loaded


## Step 3 — Load YOLO Model & Cache to HF

In [ ]:
from pathlib import Path
import shutil

from huggingface_hub import hf_hub_download
from ultralytics import YOLO

# Use YOLOv8m for better person detection than YOLOv8n.
MODEL_NAME = "yolov8m.pt"
TRACKER_NAME = f"{Path(MODEL_NAME).stem}_botsort"
LOCAL_MODEL_PATH = str(WORK_DIR / MODEL_NAME)
HF_CACHE_DIR = WORK_DIR / "hf_cache"
HF_CACHE_DIR.mkdir(parents=True, exist_ok=True)

try:
    # Try to reuse the cached model from HuggingFace.
    cached_model_path = hf_hub_download(
        repo_id=HF_REPO_ID,
        filename=MODEL_NAME,
        repo_type="dataset",
        cache_dir=str(HF_CACHE_DIR),
        token=hf_token,
    )
    shutil.copy2(cached_model_path, LOCAL_MODEL_PATH)
    print(" Model loaded from HuggingFace cache")
except Exception as cache_error:
    # First run: download the model and upload a copy.
    print(f"HuggingFace model cache miss: {cache_error}")
    model = YOLO(MODEL_NAME)
    source_model_path = Path(getattr(model, "ckpt_path", MODEL_NAME))
    if source_model_path.exists() and source_model_path.resolve() != Path(LOCAL_MODEL_PATH).resolve():
        shutil.copy2(source_model_path, LOCAL_MODEL_PATH)
    elif not Path(LOCAL_MODEL_PATH).exists():
        model.save(LOCAL_MODEL_PATH)

    api.upload_file(
        path_or_fileobj=LOCAL_MODEL_PATH,
        path_in_repo=MODEL_NAME,
        repo_id=HF_REPO_ID,
        repo_type="dataset",
        token=hf_token,
    )
    print(" Model downloaded and saved to HuggingFace")

# Load the model from the selected local path.
model = YOLO(LOCAL_MODEL_PATH)
print(f"Runtime model: {MODEL_NAME}; tracker label: {TRACKER_NAME}")
print(" Step 3 complete - YOLO model ready and cached")


 Model loaded from HuggingFace cache
 Step 3 complete — YOLO model ready and cached


## Step 4 — Run BoT-SORT Tracking

In [ ]:
from tqdm.auto import tqdm
from collections import defaultdict
import numpy as np
import cv2
import math

# Reload YOLO so BoT-SORT starts with a clean state.
model = YOLO(LOCAL_MODEL_PATH)

# Region with frequent false positives on the right side.
# It is used only during post-processing.
SUSPECT_ZONE = np.array([
    (1480, 250),
    (FRAME_WIDTH - 1, 250),
    (FRAME_WIDTH - 1, 850),
    (1480, 850),
], dtype=np.int32)

# Settings for removing static false-positive tracks.
MIN_STATIC_FRAMES = 8
MAX_STATIC_DISPLACEMENT = 45.0
MAX_STATIC_SPAN = 60.0
MIN_ZONE_HIT_RATIO = 0.5

tracking_rows_raw = []

def polygon_bbox(zone):
    """Return the axis-aligned bounding rectangle of a polygon zone."""
    xs = zone[:, 0]
    ys = zone[:, 1]
    return float(xs.min()), float(ys.min()), float(xs.max()), float(ys.max())

def bbox_overlap_ratio_with_zone(x, y, w, h, zone):
    """Measure how much of one detection box lies inside the suspect zone."""
    zx1, zy1, zx2, zy2 = polygon_bbox(zone)

    bx1, by1 = float(x), float(y)
    bx2, by2 = float(x + w), float(y + h)

    ix1 = max(bx1, zx1)
    iy1 = max(by1, zy1)
    ix2 = min(bx2, zx2)
    iy2 = min(by2, zy2)

    iw = max(0.0, ix2 - ix1)
    ih = max(0.0, iy2 - iy1)

    intersection = iw * ih
    bbox_area = max(1.0, w * h)

    return intersection / bbox_area

for frame_path in tqdm(frame_paths, desc="Tracking MOT17-02-FRCNN"):
    frame_id = int(frame_path.stem)
    frame = cv2.imread(str(frame_path))
    if frame is None:
        raise RuntimeError(f"Could not read frame: {frame_path}")

    # Use the required detector settings.
    # classes=[0] keeps only the COCO person class.
    results = model.track(
        frame,
        conf=0.3,
        iou=0.5,
        tracker="botsort.yaml",
        persist=True,
        classes=[0],
        verbose=False,
    )

    boxes = results[0].boxes if results and len(results) else None
    if boxes is None or boxes.id is None:
        continue

    # Convert YOLO outputs to NumPy arrays.
    xyxy = boxes.xyxy.detach().cpu().numpy()
    track_ids = boxes.id.detach().cpu().numpy().astype(int)
    confidences = boxes.conf.detach().cpu().numpy()
    classes = boxes.cls.detach().cpu().numpy().astype(int) if boxes.cls is not None else np.zeros(len(track_ids), dtype=int)

    for bbox, track_id, conf, cls_id in zip(xyxy, track_ids, confidences, classes):
        if cls_id != 0:
            continue

        # Convert xyxy boxes to MOT format.
        x1, y1, x2, y2 = bbox.tolist()
        w = max(0.0, x2 - x1)
        h = max(0.0, y2 - y1)

        tracking_rows_raw.append((
            frame_id,
            int(track_id),
            float(x1),
            float(y1),
            float(w),
            float(h),
            float(conf),
        ))

# Pass 1: remove static false positives near the right stall area.
track_points = defaultdict(list)

for frame_id, track_id, x, y, w, h, conf in tracking_rows_raw:
    cx = x + w / 2
    cy = y + h
    overlap_ratio = bbox_overlap_ratio_with_zone(x, y, w, h, SUSPECT_ZONE)

    track_points[track_id].append({
        "frame_id": frame_id,
        "cx": cx,
        "cy": cy,
        "in_zone": overlap_ratio > 0.2,
    })

static_false_positive_ids = set()
debug_stats = []

for track_id, points in track_points.items():
    if len(points) < MIN_STATIC_FRAMES:
        continue

    points = sorted(points, key=lambda item: item["frame_id"])

    xs = np.array([p["cx"] for p in points], dtype=float)
    ys = np.array([p["cy"] for p in points], dtype=float)
    zone_hits = sum(1 for p in points if p["in_zone"])
    zone_hit_ratio = zone_hits / len(points)

    # Check both net movement and total jitter.
    displacement = math.hypot(xs[-1] - xs[0], ys[-1] - ys[0])
    span = math.hypot(xs.max() - xs.min(), ys.max() - ys.min())

    debug_stats.append((track_id, len(points), zone_hit_ratio, displacement, span))

    is_mostly_in_suspect_zone = zone_hit_ratio >= MIN_ZONE_HIT_RATIO
    is_static = displacement <= MAX_STATIC_DISPLACEMENT and span <= MAX_STATIC_SPAN

    if is_mostly_in_suspect_zone and is_static:
        static_false_positive_ids.add(track_id)

tracking_rows = [
    row for row in tracking_rows_raw
    if row[1] not in static_false_positive_ids
]

print(f"Raw tracked person boxes: {len(tracking_rows_raw)}")
print(f"Removed static false-positive track IDs: {sorted(static_false_positive_ids)}")
print(f"Saved {len(tracking_rows)} tracked person boxes after post-processing")

# Print suspicious tracks for manual review.
print("\nDebug candidate tracks near suspect zone:")
for track_id, n_frames, zone_ratio, displacement, span in sorted(debug_stats, key=lambda x: (-x[2], x[3]))[:15]:
    print(
        f"ID {track_id}: frames={n_frames}, "
        f"zone_ratio={zone_ratio:.2f}, "
        f"displacement={displacement:.1f}, "
        f"span={span:.1f}"
    )

print("Step 4 complete - BoT-SORT tracking finished with static false-positive filtering")


Tracking MOT17-02-FRCNN:   0%|          | 0/600 [00:00<?, ?it/s]

Raw tracked person boxes: 6551
Removed static false-positive track IDs: [25, 143, 174, 181]
Saved 6317 tracked person boxes after post-processing

Debug candidate tracks near suspect zone:
ID 143: frames=11, zone_ratio=1.00, displacement=1.1, span=3.7
ID 174: frames=106, zone_ratio=1.00, displacement=5.3, span=15.2
ID 25: frames=71, zone_ratio=1.00, displacement=10.8, span=26.3
ID 181: frames=46, zone_ratio=1.00, displacement=37.7, span=54.8
ID 93: frames=35, zone_ratio=1.00, displacement=52.6, span=54.0
ID 99: frames=35, zone_ratio=1.00, displacement=92.9, span=152.3
ID 156: frames=39, zone_ratio=1.00, displacement=266.6, span=297.8
ID 3: frames=43, zone_ratio=1.00, displacement=380.5, span=380.5
ID 1: frames=61, zone_ratio=0.98, displacement=464.2, span=464.6
ID 244: frames=107, zone_ratio=0.75, displacement=642.8, span=706.4
ID 214: frames=87, zone_ratio=0.75, displacement=614.2, span=636.1
ID 80: frames=117, zone_ratio=0.74, displacement=677.0, span=707.5
ID 19: frames=64, zone_rat

In [ ]:
# Step 4.1 - Remove manually confirmed false positives.
# Keep the rule limited to the suspect region.

from collections import Counter
import numpy as np
import math

# False-positive IDs found by video review.
# Remove them only inside the suspect zone.
CONFIRMED_FALSE_POSITIVE_IDS = {44, 93, 99, 169, 196, 288}

# ID 80 starts as a stall false positive, then follows a real person.
SPECIAL_DELAYED_ID = 80

MIN_ZONE_OVERLAP = 0.20

# Set this manually if the switch frame is known.
ID80_ENABLE_FRAME_MANUAL = None

# Keep ID 80 after it moves away from the stall.
ID80_ANCHOR_FRAMES = 10
ID80_ACTIVATION_DISTANCE = 70.0
ID80_ACTIVATION_CONSECUTIVE = 3

tracking_rows_before_manual_filter = list(tracking_rows)

def row_center(row):
    """Return the bottom-center point used as a pedestrian location proxy."""
    frame_id, track_id, x, y, w, h, conf = row
    return float(x + w / 2), float(y + h)

# Estimate the early stall position of ID 80.
id80_rows = []

for row in tracking_rows_before_manual_filter:
    frame_id, track_id, x, y, w, h, conf = row
    if track_id != SPECIAL_DELAYED_ID:
        continue

    overlap_ratio = bbox_overlap_ratio_with_zone(x, y, w, h, SUSPECT_ZONE)
    if overlap_ratio >= MIN_ZONE_OVERLAP:
        cx, cy = row_center(row)
        id80_rows.append({
            "frame_id": int(frame_id),
            "cx": cx,
            "cy": cy,
            "overlap_ratio": float(overlap_ratio),
        })

id80_enable_frame = ID80_ENABLE_FRAME_MANUAL

if id80_enable_frame is None and len(id80_rows) >= ID80_ANCHOR_FRAMES:
    id80_rows = sorted(id80_rows, key=lambda item: item["frame_id"])

    # Median is less sensitive to box jitter.
    anchor_items = id80_rows[:ID80_ANCHOR_FRAMES]
    anchor_x = float(np.median([item["cx"] for item in anchor_items]))
    anchor_y = float(np.median([item["cy"] for item in anchor_items]))

    consecutive = []

    for item in id80_rows:
        dist_from_stall = math.hypot(item["cx"] - anchor_x, item["cy"] - anchor_y)

        if dist_from_stall >= ID80_ACTIVATION_DISTANCE:
            consecutive.append(item)
        else:
            consecutive = []

        if len(consecutive) >= ID80_ACTIVATION_CONSECUTIVE:
            id80_enable_frame = consecutive[0]["frame_id"]
            break

manual_removed_rows = []
tracking_rows = []

for row in tracking_rows_before_manual_filter:
    frame_id, track_id, x, y, w, h, conf = row
    overlap_ratio = bbox_overlap_ratio_with_zone(x, y, w, h, SUSPECT_ZONE)

    should_remove = False

    # Remove confirmed false positives in the suspect zone.
    if track_id in CONFIRMED_FALSE_POSITIVE_IDS and overlap_ratio > MIN_ZONE_OVERLAP:
        should_remove = True

    # For ID 80, remove early stall boxes and keep the real person.
    if track_id == SPECIAL_DELAYED_ID and overlap_ratio > MIN_ZONE_OVERLAP:
        if id80_enable_frame is None:
            should_remove = True
        elif frame_id < id80_enable_frame:
            should_remove = True

    if should_remove:
        manual_removed_rows.append(row)
        continue

    tracking_rows.append(row)

removed_by_id = Counter(row[1] for row in manual_removed_rows)

print(f"Boxes before manual filter: {len(tracking_rows_before_manual_filter)}")
print(f"Manually removed boxes: {len(manual_removed_rows)}")
print(f"Boxes after manual filter: {len(tracking_rows)}")
print(f"Removed boxes by ID: {dict(sorted(removed_by_id.items()))}")
print(f"Manual false-positive IDs: {sorted(CONFIRMED_FALSE_POSITIVE_IDS)}")
print(f"ID 80 enable frame: {id80_enable_frame}")


Boxes before manual filter: 6317
Manually removed boxes: 129
Boxes after manual filter: 6188
Removed boxes by ID: {44: 4, 80: 43, 93: 35, 99: 35, 169: 4, 196: 5, 288: 3}
Manual false-positive IDs: [44, 93, 99, 169, 196, 288]
ID 80 enable frame: 250


## Step 5 — Save Tracking Results & Upload Checkpoint

In [ ]:
from pathlib import Path

# Save predictions in MOTChallenge format.
# Format: frame,id,x,y,w,h,score,-1,-1,-1.
TRACKING_DIR = WORK_DIR / "tracking_results"
TRACKING_DIR.mkdir(parents=True, exist_ok=True)
PRED_PATH = TRACKING_DIR / "MOT17-02-FRCNN.txt"

with PRED_PATH.open("w", encoding="utf-8") as result_file:
    for frame_id, track_id, x, y, w, h, conf in sorted(tracking_rows, key=lambda row: (row[0], row[1])):
        result_file.write(f"{frame_id},{track_id},{x:.2f},{y:.2f},{w:.2f},{h:.2f},{conf:.6f},-1,-1,-1\n")

# Upload the prediction file as a checkpoint.
api.upload_file(
    path_or_fileobj=str(PRED_PATH),
    path_in_repo="checkpoints/MOT17-02-FRCNN.txt",
    repo_id=HF_REPO_ID,
    repo_type="dataset",
    token=hf_token,
)
print(" Step 5 complete - tracking results saved to HuggingFace")


 Step 5 complete — tracking results saved to HuggingFace


## Step 6 — Output Video & Upload

In [ ]:
from collections import defaultdict

# Group boxes by frame for faster rendering.
VIDEO_PATH = WORK_DIR / "output_tracking.mp4"
rows_by_frame = defaultdict(list)
for row in tracking_rows:
    rows_by_frame[row[0]].append(row)

def track_color(track_id):
    """Generate a stable display color from a track ID."""
    rng = np.random.default_rng(track_id)
    return tuple(int(channel) for channel in rng.integers(64, 256, size=3))

fourcc = cv2.VideoWriter_fourcc(*"mp4v")
writer = cv2.VideoWriter(str(VIDEO_PATH), fourcc, 30, (FRAME_WIDTH, FRAME_HEIGHT))
if not writer.isOpened():
    raise RuntimeError(f"Could not open video writer for {VIDEO_PATH}")

for frame_path in tqdm(frame_paths, desc="Rendering tracking video"):
    frame_id = int(frame_path.stem)
    frame = cv2.imread(str(frame_path))
    if frame is None:
        raise RuntimeError(f"Could not read frame: {frame_path}")

    for _, track_id, x, y, w, h, conf in rows_by_frame.get(frame_id, []):
        # Clip boxes to the image bounds.
        x1 = max(0, min(FRAME_WIDTH - 1, int(round(x))))
        y1 = max(0, min(FRAME_HEIGHT - 1, int(round(y))))
        x2 = max(0, min(FRAME_WIDTH - 1, int(round(x + w))))
        y2 = max(0, min(FRAME_HEIGHT - 1, int(round(y + h))))
        color = track_color(track_id)
        label = f"ID {track_id} {conf:.2f}"

        cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)

        # Add a label background for readability.
        label_size, baseline = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.55, 2)
        label_y1 = max(0, y1 - label_size[1] - baseline - 4)
        cv2.rectangle(frame, (x1, label_y1), (x1 + label_size[0] + 6, label_y1 + label_size[1] + baseline + 4), color, -1)
        cv2.putText(frame, label, (x1 + 3, label_y1 + label_size[1] + 1), cv2.FONT_HERSHEY_SIMPLEX, 0.55, (255, 255, 255), 2, cv2.LINE_AA)

    writer.write(frame)

writer.release()
if not VIDEO_PATH.exists() or VIDEO_PATH.stat().st_size == 0:
    raise RuntimeError(f"Video was not saved correctly: {VIDEO_PATH}")

# Upload the output video.
api.upload_file(
    path_or_fileobj=str(VIDEO_PATH),
    path_in_repo="output/output_tracking.mp4",
    repo_id=HF_REPO_ID,
    repo_type="dataset",
    token=hf_token,
)
print(" Step 6 complete - output video saved to HuggingFace")


Rendering tracking video:   0%|          | 0/600 [00:00<?, ?it/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

 Step 6 complete — output video saved to HuggingFace


## Step 7 — Evaluate: HOTA, MOTA, IDF1

In [ ]:
import shutil
import numpy as np
from pathlib import Path

# Patch old NumPy aliases used by some metric libraries.
if not hasattr(np, "asfarray"):
    np.asfarray = lambda a, dtype=float: np.asarray(a, dtype=dtype)
if not hasattr(np, "float"):
    np.float = float
if not hasattr(np, "int"):
    np.int = int
if not hasattr(np, "bool"):
    np.bool = bool

EVAL_PATH = WORK_DIR / "eval_results.txt"
TRACKER_NAME = f"{Path(MODEL_NAME).stem}_botsort"

print("Evaluation runtime check:")
print(f"  MODEL_NAME={MODEL_NAME}")
print(f"  LOCAL_MODEL_PATH={LOCAL_MODEL_PATH}")
print(f"  TRACKER_NAME={TRACKER_NAME}")
print(f"  PRED_PATH={PRED_PATH}")

pred_debug = np.empty((0, 10), dtype=float)
if PRED_PATH.exists() and PRED_PATH.stat().st_size > 0:
    pred_debug = np.loadtxt(str(PRED_PATH), delimiter=",")
    if pred_debug.ndim == 1:
        pred_debug = pred_debug.reshape(1, -1)
pred_rows = pred_debug.shape[0]
print(f"  prediction rows={pred_rows}")
if pred_rows:
    print(f"  prediction ids={len(np.unique(pred_debug[:, 1].astype(int)))}")

gt_debug = np.loadtxt(str(GT_PATH), delimiter=",")
if gt_debug.ndim == 1:
    gt_debug = gt_debug.reshape(1, -1)
valid_gt_debug = gt_debug[(gt_debug[:, 6] > 0) & (gt_debug[:, 7] == 1)]
valid_gt_rows = valid_gt_debug.shape[0]
print(f"  valid pedestrian GT rows={valid_gt_rows}")
if valid_gt_rows:
    print(f"  pred/GT row ratio={pred_rows / valid_gt_rows:.3f}")

def as_percent(value):
    """Normalize scalar or array metric values to percentage units."""
    arr = np.asarray(value, dtype=float)
    if arr.size == 0:
        return float("nan")
    score = float(np.nanmean(arr))
    return score * 100.0 if abs(score) <= 1.5 else score

def format_score(value):
    """Format numeric metric values while preserving fallback text messages."""
    if isinstance(value, str):
        return value
    if value is None or not np.isfinite(float(value)):
        return "N/A"
    return f"{float(value):.2f}%"

def run_trackeval():
    """Evaluate HOTA, MOTA, and IDF1 using the official TrackEval pipeline."""
    import trackeval
    import trackeval.datasets
    import trackeval.metrics

    # Build the folder layout expected by TrackEval.
    trackeval_root = WORK_DIR / "trackeval_data"
    if trackeval_root.exists():
        shutil.rmtree(trackeval_root)
    gt_seq_dir = trackeval_root / "gt" / "MOT17-train" / "MOT17-02-FRCNN" / "gt"
    pred_data_dir = trackeval_root / "trackers" / "MOT17-train" / TRACKER_NAME / "data"
    seqmap_dir = trackeval_root / "seqmaps"
    gt_seq_dir.mkdir(parents=True, exist_ok=True)
    pred_data_dir.mkdir(parents=True, exist_ok=True)
    seqmap_dir.mkdir(parents=True, exist_ok=True)

    # Use seqinfo.ini if available, otherwise create a minimal one.
    src_seqinfo = DATASET_DIR / "seqinfo.ini"
    dst_seqinfo = gt_seq_dir.parent / "seqinfo.ini"
    if src_seqinfo.exists():
        shutil.copy2(src_seqinfo, dst_seqinfo)
    else:
        num_frames = len(frame_paths)
        seqinfo_content = (
            "[Sequence]\n"
            "name=MOT17-02-FRCNN\n"
            f"imDir=img1\n"
            f"frameRate=30\n"
            f"seqLength={num_frames}\n"
            f"imWidth={FRAME_WIDTH}\n"
            f"imHeight={FRAME_HEIGHT}\n"
            "imExt=.jpg\n"
        )
        dst_seqinfo.write_text(seqinfo_content, encoding="utf-8")
    print(f"seqinfo.ini written to {dst_seqinfo}")

    # Copy files into TrackEval folders.
    shutil.copy2(GT_PATH, gt_seq_dir / "gt.txt")
    shutil.copy2(PRED_PATH, pred_data_dir / "MOT17-02-FRCNN.txt")
    (seqmap_dir / "MOT17-train.txt").write_text("name\nMOT17-02-FRCNN\n", encoding="utf-8")

    # Keep evaluation output short.
    eval_config = trackeval.Evaluator.get_default_eval_config()
    eval_config.update({
        "USE_PARALLEL": False,
        "PRINT_RESULTS": False,
        "PRINT_ONLY_COMBINED": True,
        "PRINT_CONFIG": False,
        "TIME_PROGRESS": False,
        "DISPLAY_LESS_PROGRESS": True,
        "OUTPUT_SUMMARY": True,
        "OUTPUT_DETAILED": False,
        "PLOT_CURVES": False,
    })

    # Evaluate the pedestrian class on MOT17 train.
    dataset_config = trackeval.datasets.MotChallenge2DBox.get_default_dataset_config()
    dataset_config.update({
        "GT_FOLDER": str(trackeval_root / "gt"),
        "TRACKERS_FOLDER": str(trackeval_root / "trackers"),
        "OUTPUT_FOLDER": str(trackeval_root / "output"),
        "TRACKERS_TO_EVAL": [TRACKER_NAME],
        "CLASSES_TO_EVAL": ["pedestrian"],
        "BENCHMARK": "MOT17",
        "SPLIT_TO_EVAL": "train",
        "INPUT_AS_ZIP": False,
        "DO_PREPROC": True,
        "TRACKER_SUB_FOLDER": "data",
        "OUTPUT_SUB_FOLDER": "",
        "SEQMAP_FOLDER": str(seqmap_dir),
        "SEQMAP_FILE": None,
        "SEQ_INFO": None,
        "GT_LOC_FORMAT": "{gt_folder}/{seq}/gt/gt.txt",
        "SKIP_SPLIT_FOL": False,
        "PRINT_CONFIG": False,
    })

    evaluator = trackeval.Evaluator(eval_config)
    dataset_list = [trackeval.datasets.MotChallenge2DBox(dataset_config)]
    metrics_list = [trackeval.metrics.HOTA(), trackeval.metrics.CLEAR(), trackeval.metrics.Identity()]
    output_res, _ = evaluator.evaluate(dataset_list, metrics_list)

    # Read the final pedestrian scores.
    dataset_key = next(iter(output_res.keys()))
    combined_res = output_res[dataset_key][TRACKER_NAME]["COMBINED_SEQ"]
    class_key = "pedestrian" if "pedestrian" in combined_res else next(iter(combined_res.keys()))
    class_res = combined_res[class_key]

    hota = as_percent(class_res["HOTA"]["HOTA"])
    mota = as_percent(class_res["CLEAR"]["MOTA"])
    idf1 = as_percent(class_res["Identity"]["IDF1"])
    return hota, mota, idf1

def load_mot_file(path):
    """Load a MOTChallenge text file and return a 10-column float array."""
    path = Path(path)
    if not path.exists() or path.stat().st_size == 0:
        return np.empty((0, 10), dtype=float)
    data = np.loadtxt(str(path), delimiter=",")
    if data.ndim == 1:
        data = data.reshape(1, -1)
    if data.shape[1] < 10:
        padded = np.full((data.shape[0], 10), -1.0, dtype=float)
        padded[:, :data.shape[1]] = data
        data = padded
    return data

def group_by_frame(data):
    """Group MOT rows by frame number for frame-by-frame metric updates."""
    groups = {}
    if data.size == 0:
        return groups
    frame_numbers = data[:, 0].astype(int)
    for frame_id in np.unique(frame_numbers):
        groups[int(frame_id)] = data[frame_numbers == frame_id]
    return groups

def run_motmetrics():
    """Fallback evaluator for MOTA and IDF1 when TrackEval is unavailable."""
    import motmetrics as mm

    gt = load_mot_file(GT_PATH)
    pred = load_mot_file(PRED_PATH)

    # Keep valid pedestrian ground-truth boxes.
    if gt.size:
        gt = gt[(gt[:, 6] > 0) & (gt[:, 7] == 1)]

    gt_by_frame = group_by_frame(gt)
    pred_by_frame = group_by_frame(pred)
    all_frames = sorted(set(gt_by_frame.keys()) | set(pred_by_frame.keys()))

    accumulator = mm.MOTAccumulator(auto_id=True)
    for frame_id in all_frames:
        gt_frame = gt_by_frame.get(frame_id, np.empty((0, 10), dtype=float))
        pred_frame = pred_by_frame.get(frame_id, np.empty((0, 10), dtype=float))

        gt_ids = gt_frame[:, 1].astype(int).tolist()
        pred_ids = pred_frame[:, 1].astype(int).tolist()
        distances = mm.distances.iou_matrix(gt_frame[:, 2:6], pred_frame[:, 2:6], max_iou=0.5)
        accumulator.update(gt_ids, pred_ids, distances)

    metrics_host = mm.metrics.create()
    summary = metrics_host.compute(accumulator, metrics=["mota", "idf1"], name="MOT17-02-FRCNN")
    mota = as_percent(summary.loc["MOT17-02-FRCNN", "mota"])
    idf1 = as_percent(summary.loc["MOT17-02-FRCNN", "idf1"])
    return "N/A (trackeval unavailable)", mota, idf1

# Use TrackEval first. Fall back to motmetrics if needed.
try:
    HOTA, MOTA, IDF1 = run_trackeval()
except Exception as trackeval_error:
    print(f"trackeval unavailable or failed; falling back to motmetrics: {trackeval_error}")
    HOTA, MOTA, IDF1 = run_motmetrics()

report = (
    "=== Tracking Evaluation Results ===\n"
    f"HOTA:  {format_score(HOTA)}\n"
    f"MOTA:  {format_score(MOTA)}\n"
    f"IDF1:  {format_score(IDF1)}\n"
)
print(report)
EVAL_PATH.write_text(report, encoding="utf-8")

# Upload the metric summary.
api.upload_file(
    path_or_fileobj=str(EVAL_PATH),
    path_in_repo="checkpoints/eval_results.txt",
    repo_id=HF_REPO_ID,
    repo_type="dataset",
    token=hf_token,
)
print(" Step 7 complete - evaluation results saved to HuggingFace")


seqinfo.ini written to /kaggle/working/trackeval_data/gt/MOT17-train/MOT17-02-FRCNN/seqinfo.ini

CLEAR Config:
THRESHOLD            : 0.5                           
PRINT_CONFIG         : True                          

Identity Config:
THRESHOLD            : 0.5                           
PRINT_CONFIG         : True                          

Evaluating 1 tracker(s) on 1 sequence(s) for 1 class(es) on MotChallenge2DBox dataset using the following metrics: HOTA, CLEAR, Identity, Count


Evaluating yolov8n_botsort

=== Tracking Evaluation Results ===
HOTA:  32.07%
MOTA:  27.25%
IDF1:  36.66%

 Step 7 complete — evaluation results saved to HuggingFace
